In [ ]:
from pathlib import Path
import os
import shutil

import pandas as pd
from IPython.display import display
from huggingface_hub import HfApi, hf_hub_download


In [ ]:
TASK_DIRNAME = "26Mar25-PyramidForcingView"
HF_ENDPOINT = os.environ.get("HF_ENDPOINT", "https://hf-mirror.com")
REPO_ID = "kv-compression/Pyramid-Forcing-Experiments"
REPO_TYPE = "dataset"
FAMILY_NAMES = [
    "causal-forcing",
    "deep-forcing",
    "pyramid-forcing",
    "rolling-forcing",
    "self-forcing",
]
DURATIONS = ["30s", "60s"]
TARGET_PREFIXES = [f"{family}-{duration}/" for family in FAMILY_NAMES for duration in DURATIONS]
VIDEO_SUFFIXES = (".mp4", ".mov", ".avi", ".mkv", ".webm")
SKIP_EXISTING = True
MANIFEST_FILENAME = "video_manifest.csv"


In [ ]:
def resolve_repo_root() -> Path:
    """Support running the notebook from repo root or from the notebook directory."""
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")


REPO_ROOT = resolve_repo_root()
DATA_DIR = REPO_ROOT / "data" / TASK_DIRNAME
VIDEO_ROOT_DIR = DATA_DIR / "videos"
MANIFEST_PATH = DATA_DIR / MANIFEST_FILENAME
DATA_DIR.mkdir(parents=True, exist_ok=True)
VIDEO_ROOT_DIR.mkdir(parents=True, exist_ok=True)
{
    "repo_root": str(REPO_ROOT),
    "video_root_dir": str(VIDEO_ROOT_DIR),
    "manifest_path": str(MANIFEST_PATH),
    "target_prefixes": TARGET_PREFIXES,
}


In [ ]:
def split_prefix(prefix: str) -> tuple[str, str]:
    name = prefix.rstrip("/")
    family, duration = name.rsplit("-", 1)
    return family, duration


def collect_target_records(repo_files: list[str]) -> pd.DataFrame:
    """Build a stable manifest for all requested family-duration prefixes."""
    records = []
    missing_prefixes = []

    for prefix in TARGET_PREFIXES:
        family, duration = split_prefix(prefix)
        matched_files = sorted(
            path
            for path in repo_files
            if path.startswith(prefix) and Path(path).suffix.lower() in VIDEO_SUFFIXES
        )
        if not matched_files:
            missing_prefixes.append(prefix)
            continue

        for video_index, repo_path in enumerate(matched_files, start=1):
            source_filename = Path(repo_path).name
            group_dir = f"{family}-{duration}"
            normalized_video_id = f"{group_dir}_video_{video_index:02d}"
            local_video_path = VIDEO_ROOT_DIR / group_dir / source_filename
            records.append(
                {
                    "family": family,
                    "duration": duration,
                    "repo_prefix": prefix,
                    "repo_path": repo_path,
                    "source_filename": source_filename,
                    "video_index": video_index,
                    "normalized_video_id": normalized_video_id,
                    "local_video_path": str(local_video_path),
                }
            )

    if missing_prefixes:
        raise FileNotFoundError(
            "No video files matched the following dataset prefixes: " + ", ".join(missing_prefixes)
        )

    manifest_df = pd.DataFrame.from_records(records)
    manifest_df = manifest_df.sort_values(
        ["family", "duration", "video_index", "source_filename"],
        kind="stable",
    ).reset_index(drop=True)
    return manifest_df


def download_target_file(api: HfApi, record: dict) -> tuple[str, str]:
    """Download one file to the workset video directory and return its local path plus status."""
    destination_path = Path(record["local_video_path"])
    destination_path.parent.mkdir(parents=True, exist_ok=True)

    if SKIP_EXISTING and destination_path.exists():
        return str(destination_path), "skipped_existing"

    cached_path = Path(
        hf_hub_download(
            repo_id=REPO_ID,
            repo_type=REPO_TYPE,
            filename=record["repo_path"],
            endpoint=HF_ENDPOINT,
        )
    )
    shutil.copy2(cached_path, destination_path)
    return str(destination_path), "downloaded"


In [ ]:
api = HfApi(endpoint=HF_ENDPOINT)
repo_files = api.list_repo_files(repo_id=REPO_ID, repo_type=REPO_TYPE)
manifest_df = collect_target_records(repo_files)

download_rows = []
for record in manifest_df.to_dict("records"):
    local_video_path, download_status = download_target_file(api, record)
    download_rows.append(
        {
            "repo_path": record["repo_path"],
            "local_video_path": local_video_path,
            "download_status": download_status,
        }
    )

manifest_df = manifest_df.drop(columns=["local_video_path"]).merge(
    pd.DataFrame(download_rows),
    on="repo_path",
    how="left",
    validate="one_to_one",
)
manifest_df.to_csv(MANIFEST_PATH, index=False)

summary_df = (
    manifest_df.groupby(["family", "duration", "download_status"], dropna=False)
    .size()
    .rename("count")
    .reset_index()
    .sort_values(["family", "duration", "download_status"], kind="stable")
    .reset_index(drop=True)
)

display(summary_df)
display(
    manifest_df[
        [
            "family",
            "duration",
            "video_index",
            "normalized_video_id",
            "source_filename",
            "download_status",
            "local_video_path",
        ]
    ]
)
print(f"Saved manifest to {MANIFEST_PATH}")
